# 01 — Exploración: estudiantes en riesgo académico

Estudiante: Maria Elizabeth Pinto Rebolledo

Prototipo del problema de **clasificación**: `at_risk` = 1 (en riesgo) o 0 (sin riesgo). Este notebook es la versión local (con pandas) del notebook de Colab. El entrenamiento que queda registrado en MLflow vive en `src/train.py`.

# Importar herramientas necesarias

In [ ]:
import pandas as pd # manejo de datos
import matplotlib.pyplot as plt # graficos
import seaborn as sns # graficos
from sklearn.model_selection import train_test_split # division de datos en entrenamiento y prueba
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler # preprocesamiento de datos
# importar modelos de clasificacion
from sklearn.linear_model import LogisticRegression # regresion logistica
from sklearn.svm import SVC # maquinas de soporte vectorial
from sklearn.tree import DecisionTreeClassifier # arboles de decision
from sklearn.ensemble import RandomForestClassifier # random forest
# importar metricas
from sklearn.metrics import accuracy_score # Metrica para clasificacion
import joblib
import json

# Cargar los datos del problema

In [ ]:
ruta_datos = "../data/final_student_dataset.csv"

df_estudiantes = pd.read_csv(ruta_datos)

df_estudiantes.head(3)

# Analisis de los datos

## 1. Revisar datos nulos

In [ ]:
df_estudiantes.info()

In [ ]:
df_estudiantes.isnull().sum()

## 2. Imputacion de datos

No se encontraron datos nulos, por lo tanto no hace falta imputar datos.

## 3. Revisar datos duplicados

In [ ]:
df_estudiantes.duplicated().sum()

In [ ]:
# Eliminar datos duplicados
df_estudiantes = df_estudiantes.drop_duplicates()
# Verificar que mis datos ya no tengan datos duplicados
df_estudiantes.duplicated().sum()

## 4. Revisar comportamiento de datos

In [ ]:
df_estudiantes.describe()

In [ ]:
# graficar columnas con valores extremos para verificar el comportamiento de los datos
# se nota que engagement_trend llega a 611 y study_load a 131

columnas_extremos = ['engagement_trend', 'study_load', 'submission_lateness_days', 'gpa_score_trend']

plt.figure(figsize=(14,10))
for i, col in enumerate(columnas_extremos):
    plt.subplot(2, 2, i+1)
    plt.hist(df_estudiantes[col], bins=30, edgecolor='black')
    plt.title('Distribucion de ' + col)
    plt.xlabel('Valor')
    plt.ylabel('Frecuencia')
    plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# graficar nube de puntos
plt.figure(figsize=(14,6))

plt.subplot(1, 2, 1)
plt.scatter(x=df_estudiantes['study_hours_per_week'], y=df_estudiantes['study_load'], edgecolor='black')
plt.title('Horas de estudio por semana vs carga de estudio')
plt.xlabel('Horas de estudio por semana')
plt.ylabel('Carga de estudio (study_load)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(x=df_estudiantes['lms_engagement_score'], y=df_estudiantes['engagement_trend'], edgecolor='black')
plt.title('Engagement en la plataforma vs tendencia de engagement')
plt.xlabel('lms_engagement_score')
plt.ylabel('engagement_trend')
plt.grid(True)

plt.tight_layout()
plt.show()

Hay pocos registros muy alejados del comportamiento general (engagement_trend hasta 611 y study_load hasta 131). Se omiten: solo se toman datos con `study_load <= 50` y `engagement_trend <= 100`.

In [ ]:
print("Filas antes del filtro: ", df_estudiantes.shape[0])

# seleccionar datos con study_load menor o igual a 50
df_estudiantes = df_estudiantes[df_estudiantes['study_load'] <= 50]
# seleccionar datos con engagement_trend menor o igual a 100
df_estudiantes = df_estudiantes[df_estudiantes['engagement_trend'] <= 100]

print("Filas despues del filtro: ", df_estudiantes.shape[0])

In [ ]:
df_estudiantes.describe()

## 5. Analizar la variable objetivo

La variable objetivo es `at_risk` (1 = en riesgo, 0 = sin riesgo), según `data_dictionary.csv`.

In [ ]:
df_estudiantes["at_risk"].value_counts()

In [ ]:
plt.figure(figsize=(6,5))
sns.countplot(data=df_estudiantes, x='at_risk')
plt.title('Distribucion de la variable objetivo at_risk')
plt.xlabel('at_risk (0 = sin riesgo, 1 = en riesgo)')
plt.ylabel('Cantidad de estudiantes')
plt.show()

Cerca de 53% y 47%: conjunto balanceado, no hace falta balancear.

## Columnas que no deben usarse para predecir

In [ ]:
correlacion = df_estudiantes.corr(numeric_only=True)['at_risk'].drop('at_risk').abs().sort_values(ascending=False)
correlacion.head(10)

`needs_help_score` y `final_score_pct` tienen correlación casi perfecta con `at_risk` (son otros objetivos del dataset). También se quitan `student_id` y `source_dataset`.

In [ ]:
columnas_eliminar = ['student_id', 'source_dataset', 'needs_help_score',
                     'needs_help_band', 'final_score_pct', 'stress_level_3class']

df_estudiantes = df_estudiantes.drop(columns=columnas_eliminar)

df_estudiantes.info()

# Preprocesamiento de los datos

## Aplicar LabelEncoder()

Todas las columnas ya son numéricas, por lo que no se transforma ninguna; se deja el código y el json igual que en el ejemplo.

In [ ]:
mapeos_encoer = {} # Definir un diccionario vacio
columna_texto = df_estudiantes.select_dtypes(include=['object', 'string']).columns

for columna in columna_texto:
    le = LabelEncoder()
    df_estudiantes[columna] = le.fit_transform(df_estudiantes[columna])
    print(f'columna {columna} transformada: {le.classes_}')

    mapeos_encoer[columna] = le.classes_.tolist()

ruta_mapeo_encoder = "mapeo_encoder_estudiantes.json"
with open(ruta_mapeo_encoder, 'w') as archivo:
    json.dump(mapeos_encoer, archivo, indent=4)

df_estudiantes.head(3)

## Aplicar StandardScaler

In [ ]:
# definir el StandardScaler()
scaler = StandardScaler()
# Entrenar el StandardScaler() con las columnas de interes.
columnas_numericas = df_estudiantes.drop(columns=columna_texto).columns
# asegurar que la columna variable objetivo no se preprocese con StandardScaler.
columnas_numericas = columnas_numericas.drop('at_risk')
scaler.fit(df_estudiantes[columnas_numericas])

# Transformar los datos con el scaler entrenado
df_estudiantes_escalado = df_estudiantes.copy()
df_estudiantes_escalado[columnas_numericas] = scaler.transform(df_estudiantes[columnas_numericas])
df_estudiantes_escalado.info()

In [ ]:
# Guardar la informacion del escalador
ruta_escalador = "scaler_estudiantes.pkl"
joblib.dump(scaler, ruta_escalador)

# Entrenamiento con datos escalados

In [ ]:
# Definir matriz "X" y objetivo "y"
X_scale = df_estudiantes_escalado.drop('at_risk', axis=1)
y_scale = df_estudiantes_escalado['at_risk']

In [ ]:
# Dividir datos en entrenamiento y prueba
X_train_scale, X_test_scale, y_train_scale, y_test_scale = train_test_split(X_scale, y_scale, test_size=0.3, random_state=42, stratify=y_scale)

print("Tamaño X_train_scale: ",X_train_scale.shape)
print("Tamaño y_train_scale: ",y_train_scale.shape)

print("Tamaño X_test_scale: ",X_test_scale.shape)
print("Tamaño y_test_scale",y_test_scale.shape)

Con más de 200 mil registros el SVC() es muy lento, por eso se entrena con una muestra aleatoria de 20.000 registros. Los demás modelos usan todos los datos.

In [ ]:
X_muestra_scale = X_train_scale.sample(20000, random_state=42)
y_muestra_scale = y_train_scale.loc[X_muestra_scale.index]

In [ ]:
# Modelo de regresion logistica
modelo_lr_s = LogisticRegression(max_iter=2000)
modelo_lr_s.fit(X_train_scale, y_train_scale)

# Modelo de maquina de soporte vectorial
modelo_svm_s = SVC()
modelo_svm_s.fit(X_muestra_scale, y_muestra_scale)

# Modelo de arbol
modelo_dt_s = DecisionTreeClassifier()
modelo_dt_s.fit(X_train_scale, y_train_scale)

# Modelo de bosques aleatorios
modelo_rf_s = RandomForestClassifier()
modelo_rf_s.fit(X_train_scale, y_train_scale)

In [ ]:
# Hacer predicciones con cada modelo y los datos de prueba
y_pred_lr_s = modelo_lr_s.predict(X_test_scale)
y_pred_svm_s = modelo_svm_s.predict(X_test_scale)
y_pred_dt_s = modelo_dt_s.predict(X_test_scale)
y_pred_rf_s = modelo_rf_s.predict(X_test_scale)

In [ ]:
# Calcular la metrica accuracy
accuracy_lr_s = accuracy_score(y_test_scale, y_pred_lr_s)
accuracy_svm_s = accuracy_score(y_test_scale, y_pred_svm_s)
accuracy_dt_s = accuracy_score(y_test_scale, y_pred_dt_s)
accuracy_rf_s = accuracy_score(y_test_scale, y_pred_rf_s)

print("Metrica accuracy de modelos")
print("Regresion logistica scale: ",accuracy_lr_s)
print("Maquina de soporte vectorial scale: ",accuracy_svm_s)
print("Arbol de decision scale: ",accuracy_dt_s)
print("Random forest scale: ",accuracy_rf_s)

## Qué debe quedar escrito al cerrar la exploración

- **Métrica** con la que `src/register.py` elegirá el mejor run: `accuracy`.
- **Features** que `src/data.py` entrega al entrenamiento: todas las columnas numéricas menos `at_risk` y las columnas eliminadas, escaladas con `StandardScaler`.
- **Límites del conjunto**: `final_student_dataset.csv` pesa ~56 MB, por eso no se sube a Git (queda en `.gitignore`).

El mejor modelo es Random Forest (accuracy ≈ 0.94). El siguiente paso es `mlflow run .`.